<a href="https://colab.research.google.com/github/l22140141/Financial_Dashboard_Modeling/blob/main/Toma_de_decisiones_basada_en_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# PRÁCTICA 1 — Análisis Exploratorio de Datos (EDA)

library(tidyverse)
library(lubridate)

# ── 1. Generar dataset simulado de ventas ─────────────────────
set.seed(123)
n <- 200

datos <- tibble(
  fecha  = seq(as.Date("2025-01-01"), as.Date("2025-12-31"), length.out = n),
  zona   = sample(c("Norte", "Sur", "Centro", "Occidente"), n, replace = TRUE),
  monto  = rnorm(n, mean = 12000, sd = 3500),
  canal  = sample(c("Online", "Tienda", "Distribuidor"), n, replace = TRUE,
                  prob = c(0.45, 0.35, 0.20))
)

# Introducir valores faltantes para practicar limpieza
datos$monto[sample(1:n, 10)] <- NA

# ── 2. Explorar estructura ────────────────────────────────────
cat("=== ESTRUCTURA DEL DATASET ===\n")
glimpse(datos)
cat("\n=== RESUMEN ESTADÍSTICO ===\n")
print(summary(datos))

# ── 3. Limpiar y transformar ──────────────────────────────────
datos_limpios <- datos |>
  drop_na(monto) |>
  mutate(
    mes       = month(fecha, label = TRUE, abbr = TRUE),
    trimestre = paste0("Q", quarter(fecha)),
    zona      = str_to_upper(zona)
  )

cat("\nRegistros originales:", nrow(datos))
cat("\nRegistros limpios:   ", nrow(datos_limpios), "\n")

# ── 4. Agregar KPIs por zona y mes ───────────────────────────
resumen <- datos_limpios |>
  group_by(zona, trimestre) |>
  summarise(
    total_ventas  = sum(monto),
    num_tickets   = n(),
    ticket_prom   = mean(monto),
    .groups       = "drop"
  ) |>
  arrange(zona, trimestre)

cat("\n=== KPIs POR ZONA Y TRIMESTRE ===\n")
print(resumen)

# ── 5. Visualización 1 — Ventas por zona y trimestre ─────────
p1 <- ggplot(resumen, aes(x = trimestre, y = total_ventas / 1000, fill = zona)) +
  geom_col(position = "dodge", width = 0.7) +
  labs(
    title    = "Ventas Totales por Zona y Trimestre",
    subtitle = "Dataset simulado — ITQ Unidad 5",
    x        = "Trimestre",
    y        = "Ventas (miles MXN)",
    fill     = "Zona"
  ) +
  theme_minimal(base_size = 13) +
  scale_fill_brewer(palette = "Set2") +
  scale_y_continuous(labels = scales::comma)

ggsave("practica1_ventas_zona.png", p1, width = 8, height = 5, dpi = 150)
cat("\n[✓] Gráfica 1 guardada: practica1_ventas_zona.png\n")

# ── 6. Visualización 2 — Distribución del ticket promedio ────
p2 <- ggplot(datos_limpios, aes(x = monto, fill = canal)) +
  geom_histogram(bins = 25, alpha = 0.75, color = "white") +
  facet_wrap(~canal) +
  labs(
    title = "Distribución de Monto de Venta por Canal",
    x     = "Monto (MXN)",
    y     = "Frecuencia"
  ) +
  theme_minimal(base_size = 12) +
  scale_fill_brewer(palette = "Pastel1") +
  theme(legend.position = "none")

ggsave("practica1_distribucion_canal.png", p2, width = 9, height = 4, dpi = 150)
cat("[✓] Gráfica 2 guardada: practica1_distribucion_canal.png\n")

# ── 7. Insight CCAIA ─────────────────────────────────────────
cat("\n=== FRAMEWORK CCAIA — INSIGHT PRINCIPAL ===\n")
top_zona <- resumen |> group_by(zona) |>
  summarise(total = sum(total_ventas)) |> arrange(desc(total)) |> slice(1)

cat("CONTEXTO  : Análisis de 190 transacciones (enero–dic 2025) en 4 zonas.\n")
cat("CONFLICTO : Existe variación significativa de ventas entre zonas.\n")
cat(sprintf("ANÁLISIS  : La zona %s lidera con $%s MXN en ventas totales.\n",
            top_zona$zona, format(round(top_zona$total), big.mark=",")))
cat("INSIGHT   : La concentración en una zona indica oportunidad de expansión.\n")
cat("ACCIÓN    : Diseñar plan de incentivos para zonas de menor desempeño en Q3.\n")


=== ESTRUCTURA DEL DATASET ===
Rows: 200
Columns: 4
$ fecha <date> 2025-01-01, 2025-01-02, 2025-01-04, 2025-01-06, 2025-01-08, 202…
$ zona  <chr> "Centro", "Centro", "Centro", "Sur", "Centro", "Sur", "Sur", "Su…
$ monto <dbl> 9513.577, 12899.093, 11136.578, 10783.601, 8669.335, 11842.403, …
$ canal <chr> "Online", "Tienda", "Online", "Online", "Online", "Distribuidor"…

=== RESUMEN ESTADÍSTICO ===
     fecha                   zona         monto             canal    
 Min.   :2025-01-01   Length   :200   Min.   : 4814   Length   :200  
 1st Qu.:2025-04-02   N.unique :  4   1st Qu.: 9783   N.unique :  3  
 Median :2025-07-02   N.blank  :  0   Median :11778   N.blank  :  0  
 Mean   :2025-07-02   Min.nchar:  3   Mean   :12043   Min.nchar:  6  
 3rd Qu.:2025-10-01   Max.nchar:  9   3rd Qu.:14196   Max.nchar: 12  
 Max.   :2025-12-31                   Max.   :23344                  
                                      NAs    :10                     

Registros originales: 200
Registros li

In [3]:
# PRÁCTICA 2 — Simulación Monte Carlo
library(tidyverse)

set.seed(42)
n_sim <- 10000

# ── 1. Simular ingresos y costos con incertidumbre ───────────
ingresos <- rnorm(n_sim, mean = 500000, sd = 80000)
costos   <- rnorm(n_sim, mean = 320000, sd = 50000)
utilidad <- ingresos - costos

# ── 2. Resultados clave ───────────────────────────────────────
cat("============================================\n")
cat("   SIMULACIÓN MONTE CARLO — 10,000 escenarios\n")
cat("============================================\n")
cat(sprintf("Utilidad media esperada : $%s MXN\n", format(round(mean(utilidad)), big.mark=",")))
cat(sprintf("Probabilidad de pérdida : %.1f%%\n",  mean(utilidad < 0) * 100))
cat(sprintf("IC 95%% inferior        : $%s MXN\n", format(round(quantile(utilidad, 0.025)), big.mark=",")))
cat(sprintf("IC 95%% superior        : $%s MXN\n", format(round(quantile(utilidad, 0.975)), big.mark=",")))
cat(sprintf("Peor escenario (1%%)    : $%s MXN\n", format(round(quantile(utilidad, 0.01)),  big.mark=",")))
cat(sprintf("Mejor escenario (99%%)  : $%s MXN\n", format(round(quantile(utilidad, 0.99)),  big.mark=",")))
cat("============================================\n")

# ── 3. Análisis de escenarios (best/base/worst) ───────────────
escenarios <- tibble(
  Escenario = c("Pesimista (P5)", "Base (P50)", "Optimista (P95)"),
  Ingresos  = round(quantile(ingresos, c(0.05, 0.50, 0.95))),
  Costos    = round(quantile(costos,   c(0.95, 0.50, 0.05))),
) |> mutate(Utilidad = Ingresos - Costos)

cat("\n=== ANÁLISIS DE ESCENARIOS ===\n")
print(escenarios)

# ── 4. Visualización — Distribución de utilidad ──────────────
resultados_df <- tibble(utilidad = utilidad)

p <- ggplot(resultados_df, aes(x = utilidad / 1000)) +
  geom_histogram(bins = 60, fill = "#0D7680", color = "white", alpha = 0.85) +
  geom_vline(xintercept = 0, color = "red", linewidth = 1.2, linetype = "dashed") +
  geom_vline(xintercept = mean(utilidad) / 1000,
             color = "#FF8C00", linewidth = 1, linetype = "solid") +
  annotate("text", x = mean(utilidad) / 1000 + 20, y = 300,
           label = paste0("Media: $", format(round(mean(utilidad)/1000), big.mark=","), "K"),
           color = "#FF8C00", size = 4, hjust = 0) +
  annotate("text", x = 15, y = 250, label = "Zona de pérdida",
           color = "red", size = 3.5) +
  labs(
    title    = "Distribución de Utilidad Esperada — Monte Carlo (n = 10,000)",
    subtitle = paste0("Prob. pérdida: ", round(mean(utilidad < 0) * 100, 2), "% | IC 95%: [$",
                      format(round(quantile(utilidad, 0.025)/1000), big.mark=","), "K, $",
                      format(round(quantile(utilidad, 0.975)/1000), big.mark=","), "K]"),
    x        = "Utilidad (miles MXN)",
    y        = "Frecuencia"
  ) +
  theme_minimal(base_size = 13)

ggsave("practica2_montecarlo.png", p, width = 9, height = 5, dpi = 150)
cat("\n[✓] Gráfica guardada: practica2_montecarlo.png\n")

# ── 5. Decisión recomendada (framework CCAIA) ─────────────────
cat("\n=== RECOMENDACIÓN DE DECISIÓN ===\n")
prob_perdida <- mean(utilidad < 0) * 100
if (prob_perdida < 5) {
  cat("DECISIÓN: PROCEDER — Probabilidad de pérdida baja (", round(prob_perdida, 1), "%)\n")
} else if (prob_perdida < 15) {
  cat("DECISIÓN: PROCEDER CON MONITOREO — Riesgo moderado (", round(prob_perdida, 1), "%)\n")
} else {
  cat("DECISIÓN: REVISAR ESTRUCTURA DE COSTOS — Riesgo elevado (", round(prob_perdida, 1), "%)\n")
}



   SIMULACIÓN MONTE CARLO — 10,000 escenarios
Utilidad media esperada : $179,054 MXN
Probabilidad de pérdida : 2.9%
IC 95% inferior        : $-7,496 MXN
IC 95% superior        : $370,234 MXN
Peor escenario (1%)    : $-39,480 MXN
Mejor escenario (99%)  : $403,065 MXN

=== ANÁLISIS DE ESCENARIOS ===
# A tibble: 3 × 4
  Escenario       Ingresos Costos Utilidad
  <chr>              <dbl>  <dbl>    <dbl>
1 Pesimista (P5)    365344 403441   -38097
2 Base (P50)        499500 320198   179302
3 Optimista (P95)   631753 236628   395125

[✓] Gráfica guardada: practica2_montecarlo.png

=== RECOMENDACIÓN DE DECISIÓN ===
DECISIÓN: PROCEDER — Probabilidad de pérdida baja ( 2.9 %)
